In [1]:
from pathlib import Path

import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import (
    CategoricalConditioningConfig,
    TimeConditioningConfig,
)
from src.monitoring.utils import save_go, scatterplot
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig
from src.stochastic_chart_transport.critic import CriticLossConfig
from src.stochastic_chart_transport.fibers import FlatFiberPacking
from src.stochastic_chart_transport.model import ChartTransportModelConfig
from src.stochastic_chart_transport.reconstruction import (
    AnchorLossConfig,
    ChartPretrainConfig,
    ReconstructionLossConfig,
)
from src.stochastic_chart_transport.study import (
    StochasticChartTransportStudyConfig,
    StochasticChartTransportStudyState,
)
from src.stochastic_chart_transport.transport import StochasticChartTransportLossConfig

device = "cuda:0"
torch_device = torch.device(device)
artifact_dir = Path("artifacts/2d-baseline")
monitor_dir = artifact_dir / "monitoring"
artifact_dir.mkdir(parents=True, exist_ok=True)
monitor_dir.mkdir(parents=True, exist_ok=True)
torch.set_float32_matmul_precision("high")

/tmp/ipykernel_1778830/3488078736.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
num_modes = 8
batch_size = 4096
ambient_dimension = 2
fiber_dimension = ambient_dimension
hidden_dim = 1024
hidden_dims = [hidden_dim] * 3

chart_pretrain_steps = 1_000
critic_pretrain_steps = 1_000
integrated_steps = 100_000
transport_chart_every_n_steps = 5
monitor_every_n_steps = 100
visualize_batch_size_per_mode = 256
generated_sample_batch_size = visualize_batch_size_per_mode * num_modes


In [3]:
mc = ChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension + fiber_dimension]
        + hidden_dims
        + [ambient_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension]
        + hidden_dims
        + [ambient_dimension + fiber_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [ambient_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
        cat_conditioning_config=CategoricalConditioningConfig(
            num_classes=2,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)

pretrain_config = ChartPretrainConfig(
    reconstruction_config=ReconstructionLossConfig(
        huber_delta=10.0,
        data_reconstruction_weight=1.0,
        prior_reconstruction_weight=1.0,
        stochastic_reconstruction_divider=10,
    ),
    anchor_config=AnchorLossConfig(
        latent_norm_weight=1e-2,
        latent_zero_mean_weight=0.1,
    ),
)

transport_config = StochasticChartTransportLossConfig(
    transport_step_multiplier=0.05,
    transport_step_cap=0.05,
    decoder_transport_weight=1.0,
    encoder_transport_weight=1.0,
    data_transport_weight=1.0,
    model_transport_weight=1.0,
    decoder_huber_delta=5.0,
    antipodal_estimate=True,
    t_range=(0.03, 0.97),
    num_time_samples=3,
)


In [4]:
data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.05,
    ambient_dimension=ambient_dimension,
    offset=0,
).to(device=torch_device)

study_config = StochasticChartTransportStudyConfig(
    data=data_config,
    prior=AnchoredGaussianScaleMixturePriorConfig.initialize(
        latent_shape=[ambient_dimension],
        precision=3.0,
    ),
    model=mc,
    pretrain=pretrain_config,
    critic=CriticLossConfig(huber_delta=10.0, weight=1.0, t_min=0.01, t_max=0.99),
    transport=transport_config,
    fiber_packing=FlatFiberPacking(fiber_ndims=fiber_dimension),
)
display(study_config.visualize())
state = StochasticChartTransportStudyState.initialize(
    config=study_config,
    device=torch_device,
)


### Chart and critic pretrain


In [5]:
with torch.cuda.device(device):
    pbar = tqdm(range(chart_pretrain_steps), desc="chart pretrain")
    for step in pbar:
        data = state.config.data.sample_unconditional(batch_size=batch_size)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = state.compute_chart_only_loss(data=data, compute_anchor_loss=True)
            total_loss = loss.sum()
        total_loss.backward()
        state.step_and_zero_grad()
        pbar.set_postfix({"loss": float(total_loss.detach())})

    pbar = tqdm(range(critic_pretrain_steps), desc="critic pretrain")
    for step in pbar:
        data = state.config.data.sample_unconditional(batch_size=batch_size)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = state.compute_critic_only_loss(data=data).sum()
        loss.backward()
        state.step_and_zero_grad()
        pbar.set_postfix({"loss": float(loss.detach())})

torch.save(state, artifact_dir / "state_pretrained.pkl")


chart pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

critic pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

### Integrated training monitor


In [6]:
PLOT_COLORS = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
]


def _cpu_float(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.detach().to(dtype=torch.float32).cpu()


def _make_segment_trace(
    *,
    base: torch.Tensor,
    field: torch.Tensor,
    color: str,
    name: str,
    showlegend: bool,
) -> go.Scatter:
    base = _cpu_float(base)
    field = _cpu_float(field)
    nan_padding = torch.full((base.shape[0], 1, 2), float("nan"))
    points = torch.cat(
        [base[:, None, :], (base + field)[:, None, :], nan_padding],
        dim=1,
    ).reshape(-1, 2)
    return go.Scatter(
        x=points[:, 0].numpy(),
        y=points[:, 1].numpy(),
        mode="lines",
        line=dict(color=color, width=1),
        opacity=0.7,
        name=name,
        legendgroup=name,
        showlegend=showlegend,
        hoverinfo="skip",
    )


def _add_scatter_panel(fig, *, data_dict, row: int, col: int, title: str):
    panel = scatterplot({k: _cpu_float(v) for k, v in data_dict.items()}, title=title)
    for trace in panel.data:
        trace.showlegend = row == 1 and col == 1
        fig.add_trace(trace, row=row, col=col)


def _add_field_panel(fig, *, base_dict, field_dict, row: int, col: int):
    for idx, (key, base) in enumerate(base_dict.items()):
        color = PLOT_COLORS[idx % len(PLOT_COLORS)]
        fig.add_trace(
            go.Scatter(
                x=_cpu_float(base)[:, 0].numpy(),
                y=_cpu_float(base)[:, 1].numpy(),
                mode="markers",
                marker=dict(size=3, color=color),
                opacity=0.35,
                name=key,
                legendgroup=key,
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            _make_segment_trace(
                base=base,
                field=field_dict[key],
                color=color,
                name=key,
                showlegend=False,
            ),
            row=row,
            col=col,
        )


def monitor(state, *, step: int) -> go.Figure:
    with torch.cuda.device(device), torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        canonical_samples = {
            str(mode_id): state.config.data.sample_class(
                mode_id=mode_id,
                batch_size=visualize_batch_size_per_mode,
            )
            for mode_id in range(num_modes)
        }
        canonical_prior = state.prior_config.sample(
            batch_size=generated_sample_batch_size
        ).to(device)
        canonical_fiber = torch.randn_like(canonical_prior)

        model_sample, _ = state.decode(canonical_prior)
        model_latent = state.encode(data=model_sample, fiber=canonical_fiber)
        scatter_data_latent = {
            key: state.encode(data=value, fiber=torch.randn_like(value))
            for key, value in canonical_samples.items()
        }
        scatter_model_latent = {"model": model_latent}

        data_transport_field = {
            key: state.config.transport._estimate_transport_field(
                state,
                data_latent=value,
                model_latent=value,
            ).chunk(2, dim=0)[0]
            for key, value in scatter_data_latent.items()
        }
        model_transport_field = {
            "model": state.config.transport._estimate_transport_field(
                state,
                data_latent=model_latent,
                model_latent=model_latent,
            ).chunk(2, dim=0)[1]
        }
        conditioned_data_transport_field = {
            key: state.config.transport._scale_clip_transport_field(value)
            for key, value in data_transport_field.items()
        }
        conditioned_model_transport_field = {
            "model": state.config.transport._scale_clip_transport_field(
                model_transport_field["model"]
            )
        }

        projected_canonical_samples = {
            key: state.config.data.project(value)
            for key, value in canonical_samples.items()
        }
        projected_model_sample = state.config.data.project(model_sample)

    combo = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=(
            "Canonical Latent Scatter",
            "Canonical Transport Field",
            "Canonical Conditioned Field",
            "Model Latent Scatter",
            "Model Conditioned Field",
            "Generated vs Canonical Samples",
        ),
    )

    _add_scatter_panel(fig=combo, data_dict=scatter_data_latent, row=1, col=1, title="")
    _add_field_panel(fig=combo, base_dict=scatter_data_latent, field_dict=data_transport_field, row=1, col=2)
    _add_field_panel(
        fig=combo,
        base_dict=scatter_data_latent,
        field_dict=conditioned_data_transport_field,
        row=1,
        col=3,
    )
    _add_scatter_panel(fig=combo, data_dict=scatter_model_latent, row=2, col=1, title="")
    _add_field_panel(
        fig=combo,
        base_dict=scatter_model_latent,
        field_dict=conditioned_model_transport_field,
        row=2,
        col=2,
    )

    sample_panel = scatterplot(
        {
            **{key: _cpu_float(value) for key, value in projected_canonical_samples.items()},
            "model": _cpu_float(projected_model_sample),
        }
    )
    for trace in sample_panel.data:
        if trace.name == "model":
            trace.marker.size = 4
            trace.opacity = 0.55
        else:
            trace.marker.size = 3
            trace.opacity = 0.35
        trace.showlegend = trace.name == "model"
        fig_name = trace.name
        trace.legendgroup = fig_name
        combo.add_trace(trace, row=2, col=3)

    combo.update_layout(
        height=900,
        width=1400,
        title=f"2D stochastic chart transport baseline, step {step}",
    )
    return combo


initial_fig = monitor(state, step=0)
display(initial_fig)
save_go(initial_fig, monitor_dir, "00000_pretrain")


## Integrated transport


In [7]:
with torch.cuda.device(device):
    pbar = tqdm(range(1, integrated_steps + 1), desc="integrated training")
    for step in pbar:
        data = state.config.data.sample_unconditional(batch_size=batch_size)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            losses = state.compute_integrated_loss(
                data=data,
                compute_transport_loss=(step % transport_chart_every_n_steps == 0),
            )
            total_loss = losses.sum()
        total_loss.backward()
        state.step_and_zero_grad()

        pbar.set_postfix({"loss": float(total_loss.detach())})
        if step % monitor_every_n_steps == 0 or step == integrated_steps:
            fig = monitor(state, step=step)
            save_go(fig, monitor_dir, f"{step:05d}")

final_fig = monitor(state, step=integrated_steps)
display(final_fig)
save_go(final_fig, artifact_dir, "final")
torch.save(state, artifact_dir / "state.pkl")

with torch.no_grad():
    final_prior = state.prior_config.sample(batch_size=generated_sample_batch_size).to(device)
    final_samples, _ = state.decode(final_prior)

torch.save(
    {
        "projected_samples": state.config.data.project(final_samples).cpu(),
        "ambient_samples": final_samples.detach().cpu(),
    },
    artifact_dir / "generated_samples.pt",
)

print(f"Saved monitoring frames to {monitor_dir}")
print(f"Saved final figure to {artifact_dir / 'final.html'}")
print(f"Saved training state to {artifact_dir / 'state.pkl'}")


integrated training:   0%|          | 0/100000 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/2d-baseline/monitoring/18500.png'